This notebook represent an attempt at predicting weaning trial failure for an invasively ventilated patient using basic machine learning techniques.
It uses a synthetic dataset that was generated using ChatGPT 5.5 with some guidance from me, who is a Registered Respiratory Therapist, with respect to some of the data I would consider useful in the context of such a prediction problem.

The notebook includes explanation of each step I take and why, as well as interpretation of results.

In [1]:
# imports

import pandas as pd

In [2]:
# read csv and store in df

df = pd.read_csv('data/synthetic_vent_weaning_trials_with_rt_notes_clean.csv')
df.head()

,patient_id,encounter_id,hospital_site,unit,room_number,rt_id,physio_id,wean_date,charting_system,age,...,failed_weaning_trial,trial_stopped_early_today,trial_stopped_reason_today,post_trial_fatigue_score,post_trial_rr,post_trial_spo2,previous_day_post_wean,trial_stopped_reason_t_minus_1,trial_stopped_reason_t_minus_2,trial_stopped_reason_t_minus_3
0,VW100001,E600763,Site_A,CVICU,525,RT_35,PT_07,2026-04-09,Cerner,47,...,Yes,Yes,fatigue,7,22.1,87.0,RR rose to ~26 bpm with visible fatigue.,anxiety,tachypnea,NaN
1,VW100002,E930323,Site_B,MICU,224,RT_39,PT_19,2026-01-12,Epic,67,...,No,Yes,copious_secretions,2,34.0,100.0,NaN,desaturation,hemodynamic_instability,tachypnea
2,VW100003,E562438,Site_B,Trauma_ICU,611,RT_31,PT_02,2026-03-01,Epic,62,...,No,Yes,desaturation,1,22.3,91.1,Pt anxious during wean; responded somewhat to ...,completed,NaN,NaN
3,VW100004,E290268,Site_A,CVICU,404,RT_18,PT_17,2026-03-18,Epic,41,...,Yes,Yes,anxiety,4,33.1,95.7,NaN,NaN,anxiety,NaN
4,VW100005,E945552,Site_A,Trauma_ICU,327,RT_02,PT_18,2026-03-29,Epic,81,...,Yes,Yes,desaturation,3,50.8,86.6,Trial stopped early yesterday due to tachypnea.,tachypnea,NaN,completed


In [4]:
df.describe()

,room_number,age,height_cm,weight_kg,bmi,frailty_score,icu_day,vent_day,days_since_first_weaning_trial,planned_wean_minutes_today,...,peep_15min,pressure_support_15min,wob_score_15min,dyspnea_borg_15min,rass_15min,actual_wean_minutes_today,wean_completion_percent_today,post_trial_fatigue_score,post_trial_rr,post_trial_spo2
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,...,1200.000000,1200.000000,1200.000000,1100.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.00000,1200.000000
mean,523.981667,66.026667,172.256083,86.642667,29.071583,4.548333,8.504167,6.168333,4.742500,118.550000,...,6.322500,9.775833,2.353333,4.274545,-0.340000,61.551667,56.263667,5.138333,27.15225,91.685500
std,201.055620,14.309959,10.997676,23.373723,6.763811,1.799638,3.971330,3.666906,3.672297,91.743728,...,2.123421,3.116410,1.330400,2.706291,1.517009,54.218358,30.358187,2.479446,8.52410,4.929886
min,201.000000,20.000000,140.000000,38.800000,16.000000,1.000000,1.000000,1.000000,0.000000,30.000000,...,0.000000,5.000000,0.000000,0.000000,-5.000000,2.000000,4.400000,0.000000,10.00000,75.500000
25%,327.000000,57.000000,164.900000,69.800000,24.300000,3.000000,5.000000,3.000000,2.000000,60.000000,...,5.000000,7.000000,1.000000,2.000000,-1.000000,25.000000,31.700000,3.000000,21.00000,88.300000
50%,519.000000,66.000000,172.100000,84.700000,29.100000,5.000000,8.000000,6.000000,4.000000,90.000000,...,6.000000,10.000000,2.000000,4.000000,0.000000,44.000000,48.300000,5.000000,27.00000,91.800000
75%,712.000000,76.000000,179.425000,100.900000,33.500000,6.000000,11.000000,9.000000,7.000000,180.000000,...,8.000000,12.000000,4.000000,6.000000,1.000000,82.000000,85.000000,7.000000,32.90000,95.500000
max,830.000000,94.000000,205.000000,197.900000,52.900000,9.000000,23.000000,21.000000,21.000000,480.000000,...,13.000000,23.000000,4.000000,10.000000,4.000000,379.000000,120.000000,10.000000,54.80000,100.000000


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Columns: 231 entries, patient_id to trial_stopped_reason_t_minus_3
dtypes: float64(106), int64(20), object(105)
memory usage: 2.1+ MB


In [12]:
df_dict = pd.read_csv('data/synthetic_vent_weaning_trials_with_rt_notes_clean_data_dictionary.csv')
df_dict.head()

,column,plain_english_meaning,timing,example_values_or_range
0,patient_id,Synthetic patient identifier.,baseline_or_administrative,varies
1,encounter_id,Synthetic hospital encounter identifier.,baseline_or_administrative,varies
2,hospital_site,Hospital site label.,baseline_or_administrative,Site_A; Site_B; Site_C
3,unit,ICU unit caring for the patient.,baseline_or_administrative,CVICU; MICU; Neuro_ICU; SICU; Trauma_ICU
4,room_number,Synthetic ICU room number.,baseline_or_administrative,201.0 to 830.0


In [19]:
unique_timing_cats = df_dict['timing'].unique()
print(f"The {len(unique_timing_cats)} unique timing categories are " + ', '.join(unique_timing_cats))

The 7 unique timing categories are baseline_or_administrative, outcome_or_after_trial, planned_trial, early_trial, prior_weaning_history, pre_trial_or_recent, target


We can see that we have a very large number of features to work with, and from the data dictionary we see that they fall into 7 categories.

It seems like we can answer a variety of questions using this dataset, so for simplicity I will focus on just one: given data about a patient in general as well as their prior weaning history, can we predict whether or not they will fail their next wean?

In [20]:
# EDA

In [ ]:
# split the data into training and test

In [ ]:
# Preprocessing

In [ ]:
# Baseline

In [ ]:
# 3 different models with cross validation and HPO

In [ ]:
# retrain best model with entire training set

In [ ]:
# test

In [ ]:
# some sample predictions

In [ ]:
# understanding how the best model makes predictions